# Metals champions — feature importance (train-only, clean split)

Champions from `selection_table.csv` — purged inner k-fold + 1SE, global cut **2021-10-06**.
All importance analysis uses **train data only** (post-split). The held-out 30% is **sealed**.

| Instrument | Group | Champion | AUC | Lower CI | Signal |
|------------|-------|----------|-----|----------|--------|
| gc1s | precious | XGB | 0.478±0.101 | 0.38 | NO |
| si1s | si1s | XGB | 0.514±0.076 | 0.44 | NO |
| pl1s | pl1s | Logistic | 0.608±0.081 | 0.53 | YES |
| hg1s | hg1s | RF | 0.604±0.041 | 0.56 | YES |

> **gc1s** and **si1s** carry no confirmed signal. gc1s runs on the pooled `precious`
> group (gc1s + si1s + pl1s); si1s and pl1s/hg1s run individually.


In [ ]:
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from IPython.display import Image, display

BASE = Path('outputs/importance')

def show(path, width=1100):
    p = Path(path)
    if p.exists():
        display(Image(str(p), width=width))
    else:
        print(f'[missing] {p}')

def load(path):
    p = Path(path)
    if p.exists():
        return pd.read_csv(p)
    print(f'[missing] {p}')
    return pd.DataFrame()

def cluster_summary(inst):
    mem = load(BASE / inst / 'cluster_membership.csv')
    mda = load(BASE / inst / 'clustered_mda_full.csv')
    if mem.empty or mda.empty:
        return pd.DataFrame()
    grp = (
        mem.groupby('cluster')
        .agg(
            n_members=('feature', 'count'),
            dominant_pfx=('f_prefix', lambda x: x.value_counts().index[0]),
            purity=('f_prefix', lambda x: round(x.value_counts().iloc[0] / len(x), 2)),
        )
        .reset_index()
    )
    return grp.merge(mda[['cluster','mean_drop','std_drop','significant']], on='cluster', how='left').sort_values('mean_drop', ascending=False)

print('Setup complete. Artifact root:', BASE.resolve())


---
## GC1S — champion: `precious` pool / XGB

**CPCV:** 15 paths · AUC 0.540 ± 0.101 · **NO SIGNAL** (lower CI 0.38)

**Significant clusters:** none.

> gc1s runs on the pooled `precious` group (gc1s + si1s + pl1s); importance scored on the gc1s slice.
> No cluster is significant; F5_signal is actually **negative** (−0.014), consistent with cl1s —
> internal signal quality features are uninformative for precious metals predictability.
> MDI–SHAP agree (τ=0.65) but MDA disagrees with both (τ≈0.21/0.24), again suggesting
> the permutation test and tree internals identify different (noisy) structure.
> The small positive raw MDA values (C2_f1, C9_f11_lowfreq_macro, C3_f2) are within noise.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('gc1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.05, vmax=0.05)
    .set_caption('gc1s — cluster summary (precious pool, K=13 corr + 4 hand-assigned = 17 groups)')
)
show(BASE / 'gc1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

No significant clusters. All std > mean.


In [ ]:
show(BASE / 'gc1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'gc1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'gc1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt = {'mda_mean': '{:.4f}'}
    for c in ['mdi_sum', 'shap_sum', 'coef_sum']:
        if c in cc.columns: fmt[c] = '{:.4f}'
    display(
        cc.style.format(fmt)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.05, vmax=0.05)
        .set_caption('gc1s — cluster cross-check (MDA · MDI · SHAP ranks)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement:')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (top 3 by raw MDA)

No significant clusters — shown for diagnostic reference.


In [ ]:
wc = load(BASE / 'gc1s' / 'within_cluster_C2_f1.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'gc1s | C2_f1 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'gc1s' / 'within_cluster_C2_f1.png', width=950)


In [ ]:
wc = load(BASE / 'gc1s' / 'within_cluster_C9_f11_lowfreq_macro.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'gc1s | C9_f11_lowfreq_macro — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'gc1s' / 'within_cluster_C9_f11_lowfreq_macro.png', width=950)


In [ ]:
wc = load(BASE / 'gc1s' / 'within_cluster_C3_f2.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'gc1s | C3_f2 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'gc1s' / 'within_cluster_C3_f2.png', width=950)


### Step 5 · Global per-feature SHAP

Mean |SHAP| per feature. No signal — illustrative only.


In [ ]:
gs = load(BASE / 'gc1s' / 'global_shap_summary.csv')
if not gs.empty:
    display(
        gs.head(25).style
        .format({'shap_magnitude': '{:.4f}', 'shap_signed': '{:.4f}', 'mdi': '{:.4f}'})
        .background_gradient(subset=['shap_magnitude'], cmap='Blues')
        .set_caption('gc1s — top 25 features by mean|SHAP| (NO SIGNAL — illustrative only)')
    )
show(BASE / 'gc1s' / 'global_shap_chart.png', width=1000)


---
## SI1S — champion: `si1s` / XGB

**CPCV:** 15 paths · AUC 0.510 ± 0.076 · **NO SIGNAL** (lower CI 0.44)

**Significant clusters:** none. All cluster MDA values are very small (< 0.008).

> si1s is arguably the least predictable instrument in the panel. AUC 0.510 is indistinguishable
> from random. No cluster achieves MDA > std. MDI–SHAP agree well (τ=0.78) but MDA agrees
> with neither (τ≈0.11/0.12) — pure noise.
> F5_signal is also slightly negative (−0.001), consistent with the broader metals finding that
> internal signal quality features provide no lift for non-predictable instruments.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('si1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.02, vmax=0.02)
    .set_caption('si1s — cluster summary (K=15 corr + 3 hand-assigned = 18 groups)')
)
show(BASE / 'si1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

No significant clusters; all drops near zero.


In [ ]:
show(BASE / 'si1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'si1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'si1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt = {'mda_mean': '{:.4f}'}
    for c in ['mdi_sum', 'shap_sum']:
        if c in cc.columns: fmt[c] = '{:.4f}'
    display(
        cc.style.format(fmt)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.02, vmax=0.02)
        .set_caption('si1s — cluster cross-check (MDA · MDI · SHAP ranks)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement:')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (top 3 by raw MDA)

Shown for structural reference only.


In [ ]:
wc = load(BASE / 'si1s' / 'within_cluster_C13_f1.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'si1s | C13_f1 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'si1s' / 'within_cluster_C13_f1.png', width=950)


In [ ]:
wc = load(BASE / 'si1s' / 'within_cluster_C2_f11.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'si1s | C2_f11 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'si1s' / 'within_cluster_C2_f11.png', width=950)


In [ ]:
wc = load(BASE / 'si1s' / 'within_cluster_C15_f2.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'si1s | C15_f2 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'si1s' / 'within_cluster_C15_f2.png', width=950)


### Step 5 · Global per-feature SHAP

Diagnostic only — AUC ≈ random (0.510).


In [ ]:
gs = load(BASE / 'si1s' / 'global_shap_summary.csv')
if not gs.empty:
    display(
        gs.head(25).style
        .format({'shap_magnitude': '{:.4f}', 'shap_signed': '{:.4f}', 'mdi': '{:.4f}'})
        .background_gradient(subset=['shap_magnitude'], cmap='Blues')
        .set_caption('si1s — top 25 features by mean|SHAP| (NO SIGNAL — illustrative only)')
    )
show(BASE / 'si1s' / 'global_shap_chart.png', width=1000)


---
## PL1S — champion: `pl1s` / Logistic (elastic-net)

**CPCV:** 15 paths · AUC 0.601 ± 0.081 · **SIGNAL** (lower CI 0.53)

**Significant clusters:** C3\_hmm\_vol only (MDA 0.017 ± 0.014).

> pl1s uses an elastic-net logistic champion. The **only significant cluster** is
> **C3_hmm_vol** — an HMM-based volatility regime cluster containing `hmm_vol_p2_turbulent`
> (|coef| 0.113) and `hmm_vol_p0_calm` (0.082). PC1 explains 65% of variance —
> a single regime latent dimension, with turbulent and calm probabilities loading on
> opposite signs. This is a regime-switching signal: the logistic model for platinum
> achieves signal by learning which volatility regime it is in.
>  
> **F5_signal is negative** (−0.020) — the largest negative F5 value of any instrument,
> meaning signal quality features actively hurt permutation-based AUC for platinum.
> MDA–Coef agreement is poor (τ=0.21), so coefficient ranks are not a reliable cross-check.
> C5_f11 has higher raw MDA (0.030) than C3_hmm_vol but with std > mean → not significant.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('pl1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.05, vmax=0.04)
    .set_caption('pl1s — cluster summary (K=14 corr + 3 hand-assigned = 17 groups)')
)
show(BASE / 'pl1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

C3_hmm_vol: only significant cluster. MDA–Coef agreement poor (τ=0.21).


In [ ]:
show(BASE / 'pl1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'pl1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'pl1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt = {'mda_mean': '{:.4f}'}
    if 'coef_sum' in cc.columns: fmt['coef_sum'] = '{:.4f}'
    display(
        cc.style.format(fmt)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.05, vmax=0.04)
        .set_caption('pl1s — cluster cross-check (MDA · Coef; elastic-net logistic)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement (MDA vs Coef):')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (significant cluster first, then top by MDA)

**C3_hmm_vol**: the only significant cluster. `hmm_vol_p2_turbulent` (|coef| 0.113)
vs `hmm_vol_p0_calm` (0.082) on opposite PC1 signs — a volatility-regime latent factor.
PC1 explains 65%: single latent dimension.


In [ ]:
wc = load(BASE / 'pl1s' / 'within_cluster_C3_hmm_vol.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'pl1s | C3_hmm_vol — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'pl1s' / 'within_cluster_C3_hmm_vol.png', width=950)


In [ ]:
wc = load(BASE / 'pl1s' / 'within_cluster_C5_f11.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'pl1s | C5_f11 — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'pl1s' / 'within_cluster_C5_f11.png', width=950)


In [ ]:
wc = load(BASE / 'pl1s' / 'within_cluster_C4_f2.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_coef_abs' if 'mean_coef_abs' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'pl1s | C4_f2 — |coef| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'pl1s' / 'within_cluster_C4_f2.png', width=950)


### Step 5 · Global per-feature coefficient

Mean |standardised coefficient| per feature, averaged across 15 CPCV paths.


In [ ]:
gc = load(BASE / 'pl1s' / 'global_coef_summary.csv')
if not gc.empty:
    display(
        gc.head(25).style
        .format({'coef_abs': '{:.4f}', 'coef_signed': '{:.4f}'})
        .background_gradient(subset=['coef_abs'], cmap='Blues')
        .set_caption('pl1s — top 25 features by mean |standardised coefficient|')
    )
show(BASE / 'pl1s' / 'global_coef_chart.png', width=1000)


---
## HG1S — champion: `hg1s` / RF

**CPCV:** 15 paths · AUC 0.604 ± 0.041 · **SIGNAL** (lower CI 0.56)

**Significant clusters:** C15\_f2 (MDA 0.042), F5\_signal (0.023), C1\_f1 (0.022) — three clusters.

> hg1s is the strongest metals champion by consistency (AUC std ±0.041, lowest in the panel).
> Three clusters clear the 1σ significance bar. **C15_f2** (volatility cluster: `f2_ret_kurt_60`,
> `f2_vol_of_vol_20`) leads with PC1=53% — a dominant but not singular vol-regime axis.
> **F5_signal and C1_f1** are also significant — unlike other metals, hg1s does benefit
> from both internal signal quality and price-action features.
> MDI–SHAP agreement is very strong (τ=0.92) but MDA disagrees with both (τ≈0.08/0.11),
> suggesting the tree's internal metrics are highly self-consistent but the permutation test
> emphasises the top 3 clusters to the exclusion of others.


### Step 1 · Feature clusters


In [ ]:
cs = cluster_summary('hg1s')
display(
    cs.style
    .format({'mean_drop': '{:.4f}', 'std_drop': '{:.4f}', 'purity': '{:.0%}'})
    .background_gradient(subset=['mean_drop'], cmap='RdYlGn', vmin=-0.02, vmax=0.05)
    .set_caption('hg1s — cluster summary (K=15 corr + 3 hand-assigned = 18 groups)')
)
show(BASE / 'hg1s' / 'dendrogram.png', width=1100)


### Step 2 · Cluster-level importance

Three significant clusters. MDI–SHAP agree (τ=0.92); MDA diverges from both.


In [ ]:
show(BASE / 'hg1s' / 'clustered_mda_chart.png', width=1050)
cc = load(BASE / 'hg1s' / 'cluster_crosscheck_table.csv')
ra = load(BASE / 'hg1s' / 'rank_agreement.csv')
if not cc.empty:
    fmt = {'mda_mean': '{:.4f}'}
    for c in ['mdi_sum', 'shap_sum']:
        if c in cc.columns: fmt[c] = '{:.4f}'
    display(
        cc.style.format(fmt)
        .background_gradient(subset=['mda_mean'], cmap='RdYlGn', vmin=-0.02, vmax=0.05)
        .set_caption('hg1s — cluster cross-check (MDA · MDI · SHAP ranks)')
    )
if not ra.empty:
    print('\nKendall \u03c4 rank agreement:')
    display(ra.style.format({'kendall_tau': '{:.2f}'}))


### Step 3 · Within-cluster breakdown (all 3 significant clusters)

**C15_f2**: `f2_ret_kurt_60` and `f2_vol_of_vol_20` dominant; PC1=53% (dominant direction).
Positive kurtosis and vol-of-vol load on positive SHAP (up-regime signal).

**F5_signal** and **C1_f1** are also significant — multi-dimensional (PC1 < 40%),
reflecting that both signal quality and price-reversal features span heterogeneous subspaces.


In [ ]:
wc = load(BASE / 'hg1s' / 'within_cluster_C15_f2.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'hg1s | C15_f2 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'hg1s' / 'within_cluster_C15_f2.png', width=950)


In [ ]:
wc = load(BASE / 'hg1s' / 'within_cluster_F5_signal.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'hg1s | F5_signal — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'hg1s' / 'within_cluster_F5_signal.png', width=950)


In [ ]:
wc = load(BASE / 'hg1s' / 'within_cluster_C1_f1.csv')
if not wc.empty:
    import math
    pc1 = wc['pca_pc1_var_explained'].iloc[0]
    pc2v = wc.get('pca_pc2_var_explained', None)
    pc2 = pc2v.iloc[0] if pc2v is not None else float('nan')
    tier = '>= 65% single latent' if pc1 >= 0.65 else ('>= 40% dominant' if pc1 >= 0.40 else '< 40% multi-dim')
    pc2_str = f' PC2={pc2:.1%}' if not math.isnan(pc2) else ''
    label = f'PC1={pc1:.1%}{pc2_str} ({tier})'
    score_col = 'mean_shap_mag' if 'mean_shap_mag' in wc.columns else wc.columns[2]
    display(
        wc[['feature', score_col, 'pc1_loading']].style
        .format({score_col: '{:.4f}', 'pc1_loading': '{:.3f}'})
        .background_gradient(subset=[score_col], cmap='Blues')
        .set_caption(f'hg1s | C1_f1 — |SHAP| ranking + PC1 loadings  ({label})')
    )
show(BASE / 'hg1s' / 'within_cluster_C1_f1.png', width=950)


### Step 5 · Global per-feature SHAP

Mean |SHAP| per feature, averaged across 15 CPCV paths. Red = positive, blue = negative.


In [ ]:
gs = load(BASE / 'hg1s' / 'global_shap_summary.csv')
if not gs.empty:
    display(
        gs.head(25).style
        .format({'shap_magnitude': '{:.4f}', 'shap_signed': '{:.4f}', 'mdi': '{:.4f}'})
        .background_gradient(subset=['shap_magnitude'], cmap='Blues')
        .set_caption('hg1s — top 25 features by mean|SHAP|')
    )
show(BASE / 'hg1s' / 'global_shap_chart.png', width=1000)


---
## Cross-instrument findings — metals

Structured summary across gc1s, si1s, pl1s, hg1s.


In [ ]:
METALS_INSTS = ['gc1s', 'si1s', 'pl1s', 'hg1s']
NOSIGNAL = {'gc1s', 'si1s'}

rows = []
for inst in METALS_INSTS:
    mda  = load(BASE / inst / 'clustered_mda_full.csv')
    ra   = load(BASE / inst / 'rank_agreement.csv')
    meta = load(BASE / inst / 'champion_meta.csv')
    if mda.empty:
        continue
    top = mda.iloc[0]
    sig_count = int(mda['significant'].sum())
    tau_1 = ra['kendall_tau'].iloc[0] if not ra.empty else float('nan')
    champion = meta['model_type'].iloc[0].upper() if not meta.empty else '?'
    signal = meta['signal'].iloc[0] if not meta.empty else False
    rows.append({
        'inst': inst,
        'champion': champion,
        'signal': '\u2713' if signal else '\u2717',
        'n_sig_clusters': sig_count,
        'top_cluster': top['cluster'],
        'top_mda': round(top['mean_drop'], 4),
        'tau_primary': round(tau_1, 2) if not pd.isna(tau_1) else 'n/a',
    })

summary = pd.DataFrame(rows).set_index('inst')
display(summary.style.set_caption('Metals champions — cross-instrument summary'))


### Key observations

**F5_signal is negative for all four metals** (gc1s: −0.014, si1s: −0.001, pl1s: −0.020,
hg1s: +0.023 **exception**). Across energy and most metals, permuting internal signal quality
features either has no effect or slightly improves AUC on test — these features carry no
incremental predictive information beyond what external market features provide (or they
introduce noise). **hg1s is the exception**: F5_signal is the second significant cluster
(MDA 0.023), suggesting copper futures have a real signal quality signal.

**pl1s** uniquely features an **HMM regime cluster** (C3_hmm_vol) as its only significant
driver — a volatility regime latent factor distinguishing turbulent from calm states.
This is qualitatively different from all other champions: platinum predictability is
regime-conditional rather than feature-level, consistent with its thin market microstructure.

**hg1s** has the most robust importance structure in metals: 3 significant clusters,
tight CPCV variance (±0.041), and strong MDI–SHAP internal agreement (τ=0.92).
Copper is the most industrially driven metal in the panel, which may explain why
its predictability comes from a broader set of features (vol regime + signal quality
+ price action) rather than a single dominant cluster.

**gc1s/si1s** show no exploitable structure — consistent with precious metals
being primarily driven by macro regime shifts not captured in the train window.
